# FindMaximumSubarray e FindMaxCrossingSubarray (Python)

[![Apri in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fabiofurini/algoritmi/blob/main/notebooks/cap17_algoritmi_ricorsivi/FindMaximumSubarray_python.ipynb)

Capitolo 17 — Algoritmi ricorsivi. Notebook eseguibile di `FindMaximumSubarray` e `FindMaxCrossingSubarray`: gli algoritmi in Python, eseguiti sugli esempi del capitolo, con la cella «Prova tu» per provarli con dati tuoi. Lo pseudocodice, gli esempi e le dimostrazioni sono nella [pagina del capitolo](https://fabiofurini.github.io/algoritmi/ricorsione/17-algoritmi-ricorsivi/).

Lo stesso notebook in [C++](https://colab.research.google.com/github/fabiofurini/algoritmi/blob/main/notebooks/cap17_algoritmi_ricorsivi/FindMaximumSubarray_cpp.ipynb) e [Java](https://colab.research.google.com/github/fabiofurini/algoritmi/blob/main/notebooks/cap17_algoritmi_ricorsivi/FindMaximumSubarray_java.ipynb); tutti gli algoritmi del capitolo nel [notebook del capitolo](https://colab.research.google.com/github/fabiofurini/algoritmi/blob/main/notebooks/cap17_algoritmi_ricorsivi_python.ipynb).

**Come si esegue.** Clic sul badge «Apri in Colab», poi *Runtime → Esegui tutto* (Ctrl+F9), oppure una cella alla volta con Maiusc+Invio, dall'alto in basso. Python 3, senza installare nulla.

**Indici.** Nello pseudocodice delle dispense gli array sono indicizzati da 1; nel codice (come in tutti i linguaggi qui usati) da 0: l'elemento `a[i]` dello pseudocodice è l'elemento di indice `i-1` nel codice, e un intervallo `a[l..r]` diventa `a[l-1..r-1]`. Le stampe dei notebook usano gli indici da 0.

**Nomi.** Le funzioni hanno il nome dello pseudocodice, nella convenzione del linguaggio (come nei capitoli di introduzione): `RecursiveSum` → `recursive_sum` (snake_case).

## FindMaxCrossingSubarray e FindMaximumSubarray (divide et impera)

- **FindMaxCrossingSubarray**$(a,l,mid,r)$ — *Input*: un array e tre indici $l\le mid<r$. *Output*: gli indici $maxleft\le mid<maxright$ di un sottoarray di somma massima fra quelli che attraversano il punto medio, e la sua somma. Scandisce la metà sinistra da $mid$ a $l$ e la destra da $mid+1$ a $r$: $\Theta(n)$.
- **FindMaximumSubarray**$(a,l,r)$ — *Input*: un array e due indici $l\le r$. *Output*: gli indici $i\le j$ di un sottoarray di somma massima di $a[l..r]$ e la sua somma. Il migliore fra il sottoarray massimo della metà sinistra, della destra e di quelli che attraversano il punto medio (a parità, prima il sinistro, poi il destro). $S(n)=2S(n/2)+\Theta(n)=\Theta(n\log_2 n)$, $2n-1$ chiamate.

Il $-\infty$ dello pseudocodice è il valore più piccolo rappresentabile. Prima la traccia sull'esempio $a=[-2,5,-1,3]$ (come l'esecuzione dell'implementazione C++ del capitolo; risultato $(1,3,7)$, cioè $(2,4,7)$ con indici da 1), poi l'esempio della compravendita di azioni, con le variazioni giornaliere $a=[13,-3,-25,20,-3,-16,-23,18,20,-7,12,-5,-22,15,-4,7]$: tabella dei risultati delle chiamate sui sottointervalli (per livello dell'albero, come nel capitolo; gli indici sono da 0) e i calcoli dei due `FindMaxCrossingSubarray` del livello 1. Il risultato è $(7,10,43)$, cioè $(8,11,43)$ con indici da 1.

In [1]:
fms_calls = 0   # chiamate di FindMaximumSubarray

def find_max_crossing_subarray(a, l, mid, r):
    """Restituisce (maxleft, maxright, somma) del miglior sottoarray che attraversa mid."""
    leftsum = float("-inf")
    s = 0
    maxleft = mid
    for i in range(mid, l - 1, -1):         # da mid a l
        s = s + a[i]                        # somma di a[i..mid]
        if s > leftsum:
            leftsum = s
            maxleft = i
    rightsum = float("-inf")
    s = 0
    maxright = mid + 1
    for j in range(mid + 1, r + 1):         # da mid+1 a r
        s = s + a[j]                        # somma di a[mid+1..j]
        if s > rightsum:
            rightsum = s
            maxright = j
    return (maxleft, maxright, leftsum + rightsum)

def find_maximum_subarray(a, l, r):
    """Restituisce (i, j, somma) di un sottoarray di somma massima di a[l..r]."""
    global fms_calls
    fms_calls += 1
    if l == r:
        return (l, r, a[l])                 # caso base: un elemento
    mid = l + (r - l) // 2
    ll, lr, lsum = find_maximum_subarray(a, l, mid)
    rl, rr, rsum = find_maximum_subarray(a, mid + 1, r)
    cl, cr, csum = find_max_crossing_subarray(a, l, mid, r)
    if lsum >= rsum and lsum >= csum:
        return (ll, lr, lsum)
    if rsum >= lsum and rsum >= csum:
        return (rl, rr, rsum)
    return (cl, cr, csum)

# ---- varianti che stampano la traccia ----
def fms_trace(a, l, r, depth):
    pad = "  " * depth
    if l == r:
        print(f"{pad}FindMaximumSubarray(a, {l}, {r}) = ({l}, {r}, {a[l]})  caso base")
        return (l, r, a[l])
    mid = l + (r - l) // 2
    print(f"{pad}FindMaximumSubarray(a, {l}, {r}): l != r, mid = {mid}")
    left = fms_trace(a, l, mid, depth + 1)
    right = fms_trace(a, mid + 1, r, depth + 1)
    cross = find_max_crossing_subarray(a, l, mid, r)
    print(f"{pad}  FindMaxCrossingSubarray(a, {l}, {mid}, {r}) = {cross}")
    if left[2] >= right[2] and left[2] >= cross[2]:
        ret = left
    elif right[2] >= left[2] and right[2] >= cross[2]:
        ret = right
    else:
        ret = cross
    print(f"{pad}  left.sum={left[2]}, right.sum={right[2]}, cross.sum={cross[2]} => restituisce {ret}")
    return ret

def fms_rows(a, l, r, rows):
    if l == r:
        return (l, r, a[l])
    mid = l + (r - l) // 2
    left = fms_rows(a, l, mid, rows)
    right = fms_rows(a, mid + 1, r, rows)
    cross = find_max_crossing_subarray(a, l, mid, r)
    if left[2] >= right[2] and left[2] >= cross[2]:
        ret = left
    elif right[2] >= left[2] and right[2] >= cross[2]:
        ret = right
    else:
        ret = cross
    rows.append((l, r, mid, left, right, cross, ret))
    return ret

def crossing_trace(a, l, mid, r):
    print(f"FindMaxCrossingSubarray(a, {l}, {mid}, {r})")
    print(f"{'i':>3s} {'a[i]':>5s} {'sum':>5s} {'leftsum':>8s} {'maxleft':>8s}")
    leftsum = float("-inf")
    s = 0
    for i in range(mid, l - 1, -1):
        s = s + a[i]
        if s > leftsum:
            leftsum, maxleft = s, i
        print(f"{i:>3d} {a[i]:>5d} {s:>5d} {leftsum:>8d} {maxleft:>8d}")
    print(f"{'j':>3s} {'a[j]':>5s} {'sum':>5s} {'rightsum':>8s} {'maxright':>8s}")
    rightsum = float("-inf")
    s = 0
    for j in range(mid + 1, r + 1):
        s = s + a[j]
        if s > rightsum:
            rightsum, maxright = s, j
        print(f"{j:>3d} {a[j]:>5d} {s:>5d} {rightsum:>8d} {maxright:>8d}")
    print(f"sottoarray che attraversa a[{maxleft}..{maxright}], somma {leftsum} + {rightsum} = {leftsum + rightsum}")

a = [-2, 5, -1, 3]
print("a =", a)
res = fms_trace(a, 0, len(a) - 1, 0)
fms_calls = 0
find_maximum_subarray(a, 0, len(a) - 1)
print(f"risultato {res}   ({res[0] + 1}, {res[1] + 1}, {res[2]} con indici da 1)   chiamate={fms_calls} (2n-1={2 * len(a) - 1})")

print()
a = [13, -3, -25, 20, -3, -16, -23, 18, 20, -7, 12, -5, -22, 15, -4, 7]
n = len(a)
print("a =", a)
fms_calls = 0
res = find_maximum_subarray(a, 0, n - 1)
print(f"FindMaximumSubarray(a, 0, {n - 1}) = {res}   ({res[0] + 1}, {res[1] + 1}, {res[2]} con indici da 1)   chiamate={fms_calls} (2n-1={2 * n - 1})")
rows = []
fms_rows(a, 0, n - 1, rows)
rows.sort(key=lambda t: (-(t[1] - t[0]), t[0]))
print(f"{'call':<9s} {'mid':>3s}  {'left':<13s} {'right':<13s} {'crossing':<13s} restituito")
for (l, r, mid, left, right, cross, ret) in rows:
    print(f"{'[' + str(l) + '..' + str(r) + ']':<9s} {mid:>3d}  {str(left):<13s} {str(right):<13s} {str(cross):<13s} {ret}")
print()
crossing_trace(a, 0, 3, 7)
print()
crossing_trace(a, 8, 11, 15)

a = [-2, 5, -1, 3]
FindMaximumSubarray(a, 0, 3): l != r, mid = 1
  FindMaximumSubarray(a, 0, 1): l != r, mid = 0
    FindMaximumSubarray(a, 0, 0) = (0, 0, -2)  caso base
    FindMaximumSubarray(a, 1, 1) = (1, 1, 5)  caso base
    FindMaxCrossingSubarray(a, 0, 0, 1) = (0, 1, 3)
    left.sum=-2, right.sum=5, cross.sum=3 => restituisce (1, 1, 5)
  FindMaximumSubarray(a, 2, 3): l != r, mid = 2
    FindMaximumSubarray(a, 2, 2) = (2, 2, -1)  caso base
    FindMaximumSubarray(a, 3, 3) = (3, 3, 3)  caso base
    FindMaxCrossingSubarray(a, 2, 2, 3) = (2, 3, 2)
    left.sum=-1, right.sum=3, cross.sum=2 => restituisce (3, 3, 3)
  FindMaxCrossingSubarray(a, 0, 1, 3) = (1, 3, 7)
  left.sum=5, right.sum=3, cross.sum=7 => restituisce (1, 3, 7)
risultato (1, 3, 7)   (2, 4, 7 con indici da 1)   chiamate=7 (2n-1=7)

a = [13, -3, -25, 20, -3, -16, -23, 18, 20, -7, 12, -5, -22, 15, -4, 7]
FindMaximumSubarray(a, 0, 15) = (7, 10, 43)   (8, 11, 43 con indici da 1)   chiamate=31 (2n-1=31)
call      mid  left 

**Prova tu.** Cambia l'array: il sottoarray massimo può avere un solo elemento o attraversare il punto medio; le chiamate sono sempre $2n-1$.

In [2]:
# Cambia l'array: quanti elementi ha il sottoarray massimo? Le chiamate sono sempre 2n-1.
b = [4, -1, 2, 1, -5, 4, -7, 3, 3]
fms_calls = 0
print(find_maximum_subarray(b, 0, len(b) - 1), fms_calls)

(0, 3, 6) 17


---

Materiale didattico di [Fabio Furini](https://fabiofurini.github.io/) — DIAG, Sapienza Università di Roma.
Testo, figure e dati [CC BY 4.0](https://github.com/fabiofurini/algoritmi/blob/main/LICENSE), codice [MIT](https://github.com/fabiofurini/algoritmi/blob/main/LICENSE-CODE).